[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/electrokinetics/electrophoresis.ipynb)

# Capillary Electrophoresis

This example computes how neutral and charged dye bands travel along a triangulated channel between charged walls under a pressure drop or an electric field, illustrating the separation mechanism of capillary electrophoresis. Negatively charged walls draw positive counter-ions into thin screening clouds beside them, which are called electric double layers. Driving the liquid with a pressure drop pushes the entire cross-section while no-slip walls hold the boundary still, shearing a transverse dye band into a Poiseuille parabola. Applying an axial electric field pulls only the thin charged clouds beside the walls, which drag the neutral interior along as a flat electro-osmotic plug so that a neutral band stays compact even as the flow accelerates through a midpoint constriction. Charged dye species also drift relative to the liquid under the electric field, splitting positive, neutral, and negative markers into distinct bands.

Planar projective spacetime places channel vertices at unit projective weight along `origin`, joined with a degenerate quantity direction `v` for transported fluid volume. The term ***parcel*** here means a material element carrying extensive contents without projective weight, so a unit volume of liquid is modelled as `volume` plus ionic charge along `time`, whereas a dye molecule is modelled as pure charge `charge * time` without volume. Under a combined pressure and electric drive `-(gradient ^ (pressure * v + potential * t))`, `drive.commutator(charge * time)` pushes charged dyes in proportion to their valence while passing over neutral dyes and ignoring pressure gradients completely. Pairing triangle drift velocities against each corner's simplex gradient transfers dye conservatively between adjacent dual cells so total dye mass stays invariant. [The electrokinetic transport notebook](electrokinetics.ipynb) develops the underlying double-layer, reciprocity, and entropy-production structure across wavy channels and power-generating slits.

In vector-calculus notation, species transport writes advection, electrophoretic drift, and Fickian diffusion as $\partial_t c + \nabla\cdot\big((\mathbf u - z\mu\nabla\phi)\,c - D\nabla c\big) = 0$, paired in a wide slit of half-width $h$ with Poiseuille's pressure-driven parabola $u(y) = \tfrac32 U\,(1 - y^2/h^2)$ and the Helmholtz–Smoluchowski plug speed $u = -\epsilon\zeta E/\eta$ outside thin wall double layers of zeta potential $\zeta$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import SparseExtensor, spdiag
from numga.topology import Topology
from examples.util.animation import show_states
from examples.fluids.electrokinetics import render, scenarios
from examples.fluids.electrokinetics.numerical import coupled_solution

# Planar projective spacetime for channel geometry and electrodynamics, joined with volume as a transported quantity.
spacetime = Algebra("x+y+t-w0")
quantities = Algebra("v0")
ga = spacetime * quantities
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Antiscalar = ga.gatype.antiscalar()                                           # Antiscalar-
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
Upright = Plane.derive.within("x y")                                          # Vector+
Drive = ga.gatype.bivector().derive.touching("x y", "t v")                    # Bivector+
Flux = ga.gatype.antibivector()                                               # Antibivector-
origin = mv.w.dual()                                                          # [] Point-
axis = mv.x.dual()                                                            # [] Point-
across = mv.y.dual()                                                          # [] Point-
time = -mv.t.dual()                                                           # [] Point-
volume = mv.v.dual()                                                          # [] Point-
unread = origin & time & volume                                               # [] Antitrivector-
pump, battery = mv.xv, mv.xt                                                  # [] Drive+, Drive+

## Pressure-driven and electro-osmotic channel flows

Uniformly charged walls bound a periodic channel many Debye lengths wide that narrows smoothly to a midpoint neck. The term ***simplex gradient*** here means the spatial plane for a triangle corner that vanishes along the facing side and rises by one to that corner, `(unread & sides) / double_areas`. Damped Newton iteration settles the Boltzmann counter-ion clouds against dielectric outflow `-(gradients ^ t)`, after which coupled Stokes vorticity and charge conservation determine the flow and induced potential under `pump` and `battery`, each normalized to unit mean speed `(pump & flux) / width`. Settling the double layer and solving both channel flows takes about ten seconds.

In [ ]:
topology, places, edge_shifts, lower_wall, upper_wall, wall_charge_values, wall_charges = scenarios.capillary(
    scenarios.neck_swing)
valences, bulk, mobilities = scenarios.valences, scenarios.bulk, scenarios.mobilities
viscosity, n_newton_steps = scenarios.viscosity, scenarios.n_newton_steps

In [ ]:
vertices = (origin + axis * places[..., 0] + across * places[..., 1]).field()  # Point-[V]
edges = topology.d[0] * vertices + (axis * edge_shifts).field()                # Point-[E]
sides = topology.boundary[2, 1] * edges[..., None]                             # [3] Point-[F]
double_areas = sides[..., 0] & sides[..., 1] & unread                          # Antiscalar-[F]
gradients = (unread & sides) / double_areas[..., None]                         # [3] Upright+[F]
areas = double_areas.norm() / 2                                                # Scalar+[F]
corners = topology.incidence[2, 0]                                             # [3] Scalar+[F <- V]
dual_areas = corners.sum(axis=0).adjugate() * (areas / 3)                      # Scalar+[V]


def vertex_form(corner_form: Scalar) -> SparseExtensor:
    """Pullback by triangle-corner incidence assembles a local `[..., 3, 3]` form (`Scalar+[F]`) into a vertex operator (`[...] Scalar+[V <- V]`)."""
    return (corners[:, None].adjugate() * spdiag(corner_form) * corners[None, :]).sum(axis=-1).sum(axis=-1)


def double_layer(outflow: SparseExtensor, wall_charge_field: Scalar) -> Scalar:
    """Damped Newton iteration solves the Poisson-Boltzmann balance between dielectric `outflow` and ionic plus `wall_charge_field` (`Scalar+[V]`)."""
    potential_field = 0 * wall_charge_field                                    # Scalar+[V]
    for _ in range(n_newton_steps):
        ion_densities = bulk * (-valences * potential_field[..., None]).exp()  # [2] Scalar+[V]
        residual = outflow * potential_field - dual_areas * (valences * ion_densities).sum(axis=-1) - wall_charge_field
        susceptibility = (valences ** 2 * ion_densities).sum(axis=-1)          # Scalar+[V]
        step = ((outflow + dual_areas * susceptibility) * Scalar).solve(residual)  # Scalar+[V]
        potential_field = potential_field - step.tanh()
    return potential_field


corner_drives = -(gradients ^ mv.t)                                            # [3] Drive+[F]
excitations = corner_drives.commutator(time) & time                            # [3] Flux-[F]
outflow = vertex_form((corner_drives[:, None] & excitations[None, :]) * areas)
potential = double_layer(outflow, mv.scalar(wall_charge_values[:, None]).field())  # Scalar+[V]

In [ ]:
experiments = stack((pump, battery))                                           # [2] Drive+
concentrations = corners.mean(axis=0) * (bulk * (-valences * potential[..., None]).exp())  # [2] Scalar+[F]
parcels = volume + (time * valences * concentrations).sum(axis=-1)             # Point-[F]
conductivity = (mobilities * valences ** 2 * concentrations).sum(axis=-1)      # Scalar+[F]
flows = sides / 2                                                              # [3] Point-[F]
corner_currents = corner_drives.commutator(time) & time                        # [3] Flux-[F]
circulation_plane = (stack((mv.x, mv.y)) * (stack((mv.x, mv.y)) & Point)).sum(axis=0)  # [] Plane+ <- Point-
corner_velocities = flows / areas[..., None]                                   # [3] Point-[F]
circulation = vertex_form((circulation_plane(corner_velocities[:, None]) & corner_velocities[None, :]) * areas)
viscous = viscosity * circulation * spdiag(1 / dual_areas) * circulation       # Scalar+[V <- V]
advected_fluxes = flows & parcels[..., None]                                   # [3] Flux-[F]
coupling = vertex_form(corner_drives[None, :] & advected_fluxes[:, None])
conduction = vertex_form((corner_drives[:, None] & corner_currents[None, :]) * (conductivity * areas))
flow_load = (corners.adjugate() * (experiments[:, None] & advected_fluxes)).sum(axis=-1)   # [2] Scalar+[V]
applied_current = conductivity * (experiments.commutator(time) & time)         # [2] Flux-[F]
current_load = (corners.adjugate() * ((corner_drives & applied_current[:, None]) * areas)).sum(axis=-1)
stream, induced_potentials = coupled_solution(
    viscous, coupling, conduction, flow_load, current_load, topology.interior[0], upper_wall, int(lower_wall[0]))
velocities = ((corners * stream[..., None]) * sides).sum(axis=-1) / (2 * areas)            # [2] Point-[F]
field = experiments + ((corners * induced_potentials[..., None]) * corner_drives).sum(axis=-1)  # [2] Drive+[F]
channel_fluxes = ((velocities & parcels) * areas).sites.sum() / scenarios.capillary_length # [2] Flux-
mean_speeds = (pump & channel_fluxes) / (2 * scenarios.capillary_half_width)   # [2] Scalar+
velocities, field = velocities / mean_speeds, field / mean_speeds              # [2] Point-[F], Drive+[F]
render.profiles_figure(topology, vertices, scenarios.n_capillary_across, velocities);

Across the widest station of the channel, the orange curve shows the pressure-driven velocity profile normalized by its cross-sectional mean, peaking at three halves of the mean on the centreline and vanishing at the no-slip walls as a Poiseuille parabola. The blue curve shows the field-driven electro-osmotic profile, which stays flat across the neutral core and drops to zero only inside the thin positive counter-ion clouds beside the walls.

## Electrophoretic separation of dye bands

Releasing a Gaussian dye band across the channel compares neutral advection under pressure driving against simultaneous neutral, positive, and negative transport under electric-field driving. Each dye species drifts relative to the liquid by `dye_mobility * field.commutator(charges * time)` while spreading slowly by Fickian diffusion `gradients & fick_flux(gradients)`. Because triangle simplex gradients sum to zero on every element, pairing triangle drift against `gradients` transfers dye conservatively from one dual cell to its neighbours. Stepping and rendering the dye animation takes about ten seconds.

In [ ]:
band_middle, band_width = 6.0, 0.8
dye_mobility, dye_diffusivity = 0.4, 0.02
run_drives, dye_charges = np.array([0, 1, 1, 1]), np.array([0, 0, 1, -1])
n_runs = len(dye_charges)
dye_dt, n_dye_frames, dye_stride = 0.1, 100, 3
frame_duration_ms = 40

In [ ]:
fick_flux = (stack((axis, across)) * (Plane & stack((axis, across)))).sum(axis=0)  # [] Point- <- Plane+


def dye_transport(flow_velocities: Point, total_field: Drive, charges: np.ndarray, dye: Scalar) -> Iterator[Scalar]:
    """Yields Crank-Nicolson frames of `dye` (`[n_runs] Scalar+[V]`) carried by `flow_velocities` (`[n_runs] Point-[F]`), drifting under `total_field` by `charges`, and diffusing."""
    drifts = flow_velocities + dye_mobility * total_field.commutator(charges * time)  # [n_runs] Point-[F]
    corner_inflows = areas[..., None] * (gradients & drifts[..., None])        # [n_runs, 3] Scalar+[F]
    advection = vertex_form(corner_inflows[..., :, None] * np.full(3, 1 / 3))  # [n_runs] Scalar+[V <- V]
    overlap_form = vertex_form(areas[..., None, None] * (np.ones((3, 3)) + np.eye(3)) / 12)
    diffusion = vertex_form(areas[..., None, None] * (gradients[:, None] & fick_flux(gradients[None, :])))
    transport_rate = dye_diffusivity * diffusion - advection                   # [n_runs] Scalar+[V <- V]
    forward_step = overlap_form / dye_dt - transport_rate / 2                  # [n_runs] Scalar+[V <- V]
    backward_step = (overlap_form / dye_dt + transport_rate / 2) * Scalar      # [n_runs] (Scalar+ <- Scalar+)[V <- V]
    yield dye
    for _ in range(n_dye_frames):
        for _ in range(dye_stride):
            dye = backward_step.solve(forward_step * dye)
        yield dye


axial_distances = mv.x & vertices                                              # Scalar+[V]
band = np.ones(n_runs) * (-((axial_distances - band_middle) / band_width).squared() / 2).exp()  # [n_runs] Scalar+[V]
density = (valences * bulk * (-valences * potential[..., None]).exp()).sum(axis=-1)             # Scalar+[V]
vertex_velocities = (corners.sum(axis=0).adjugate() * (velocities * (areas / 3))) / dual_areas  # [2] Point-[V]
axial_fields = mv.x & experiments.commutator(time)                             # [2] Scalar+
drive_potentials = (induced_potentials - axial_fields * axial_distances) / mean_speeds          # [2] Scalar+[V]
falls = axial_fields * scenarios.capillary_length / mean_speeds                # [2] Scalar+
dye_states = show_states(
    dye_transport(velocities[run_drives], field[run_drives], dye_charges, band),
    render.dyes_animation,
    topology, vertices, scenarios.n_capillary_across, scenarios.capillary_length,
    wall_charges, density, vertex_velocities, drive_potentials, falls,
    name="electrophoresis", duration_ms=frame_duration_ms,
)

Green strokes mark the negatively charged walls, faint magenta shading along them marks the positive counter-ion double layers, grey streaks indicate fluid speed, and dark equipotential contours crowd through the midpoint neck in the field-driven lower strip while remaining absent in the pressure-driven upper strip. In the upper strip, parabolic Poiseuille shear stretches the blue neutral band into a long tongue with slow tails trapped near the walls. In the lower strip, electro-osmotic plug flow carries the blue neutral marker intact through the neck while electrophoretic drift drives the magenta positive band ahead of the liquid and holds the green negative band behind it.

In [ ]:
# checks
# Pressure-driven flow peaks at three halves of the local mean speed through the midpoint neck.
peak_speed = (mv.x & velocities[0]).max()
neck_half_width = scenarios.capillary_half_width - scenarios.neck_swing
expected_peak = 1.5 * scenarios.capillary_half_width / neck_half_width
assert (peak_speed - expected_peak).abs() < 0.05 * expected_peak
# Dual-cell dye totals stay conserved across every transport run.
final_dye = dye_states[-1]
initial_totals = (dual_areas * band).sites.sum()                               # [n_runs] Scalar+
assert (((dual_areas * final_dye).sites.sum() - initial_totals).abs() < 1e-10 * initial_totals).all()
# Electric field separates positive, neutral, and negative dye centroids in valence order.
mean_shifts = (dual_areas * axial_distances * (final_dye - band)).sites.sum() / initial_totals  # [n_runs] Scalar+
assert (mean_shifts[2] > mean_shifts[1]) and (mean_shifts[1] > mean_shifts[3])